In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new-G14.csv") #-data.csv")

In [4]:
df_ground_truth.head()

,question,document
0,I just stumbled upon this course. Am I still a...,74eb249bbf
1,"If I enroll now, will I be eligible for a cert...",74eb249bbf
2,What's the cutoff date for project submissions...,74eb249bbf
3,I missed the beginning of the course. Can I st...,74eb249bbf
4,Is it possible to join the course late and sti...,74eb249bbf


In [ ]:
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth

In [ ]:
ground_truth[10]

In [8]:
from ingest import load_faq_data, build_index

documents = load_faq_data(file_path="../documents/all_documents.json")

documents_llm = []

for doc in documents:
    if doc["course"] == "llm":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [9]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'course': 'llm',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.',
  'doc_id': 'db78580409'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM  docs](https://site.club/docs/courses/llm/), the [general  logistics docs](https://site.club/docs/courses/logistics/), and the [LLM  GitHub re

In [10]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [11]:
q = ground_truth[0]
q

{'question': "I just stumbled upon this course. Am I still allowed to join even though it's already started?",
 'document': '74eb249bbf'}

In [12]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [30]:
q

{'question': "What's the main reason the course decided to use uv instead of the traditional pip and virtualenv setup?",
 'document': 'f81dea8f7e'}

In [14]:
results = text_search(q['question'])
results

[{'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.",
  'doc_id': 'a9353fadfe'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?',
  'answer': 'When you set up your account, you are automatical

In [27]:
for d in results:
    print(f'{d["doc_id"]} --- {d["question"]}')

74eb249bbf --- I just discovered the course. Can I still join?
a9353fadfe --- The homework submission form is still open even though the deadline has passed — can I still submit?
c2903069a0 --- Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
9f689c185f --- I missed the first homework - can I still get a certificate?
85384a18e5 --- OpenAI: Do I have to subscribe and pay for Open AI API for this course?


In [ ]:
for d in results:
    print(f'{d["doc_id"]} == {doc_id}: {d["doc_id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
c2903069a0 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False


In [17]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [31]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [38]:
doc, doc_id

({'course': 'machine-learning',
  'section': 'Miscellaneous',
  'question': "My homework answer doesn't match any of the options",
  'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it.",
  'doc_id': 'ab183bd688'},
 '74eb249bbf')

In [32]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)


I just stumbled upon this course. Am I still allowed to join even though it's already started?


[1, 0, 0, 0, 0]

In [36]:
q = ground_truth[121]
print(q)
compute_relevance_text(q)


{'question': "What's the proper way to store API keys like OpenAI or Groq in a project? Should I put them in a .env file?", 'document': '233dabe430'}


[1, 0, 0, 0, 0]

In [ ]:
[0, 0, 1, 0, 0]

In [43]:
q = ground_truth[12]
print(q)
compute_relevance_text(q)

{'question': 'How are we supposed to ask questions during the live stream—should we use Slido?', 'document': '489dd1c9d9'}


[1, 0, 0, 0, 0]

In [ ]:
[0, 0, 0, 0, 0]

In [45]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

/Users/sithvothy.kiv/Downloads/04-evaluation-G14/qwen_version/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [47]:
len(ground_truth)

556

In [48]:
relevance = compute_relevance_total_text(ground_truth)

100%|██████████| 556/556 [00:00<00:00, 741.60it/s]


In [49]:
relevance[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [50]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

In [ ]:
sample = relevance_total[:15]

In [ ]:
sample

In [ ]:
14 / 15

In [ ]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
hit_rate(relevance)

In [ ]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [ ]:
mrr(sample)

In [ ]:
mrr(relevance)

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(ground_truth, text_search)

In [ ]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
evaluate(ground_truth, text_search_v2)

In [ ]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

In [ ]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)